# Pythia 410m: Accessibility Domain Knowledge

24 layers | 16 heads | d_model=1024 | d_mlp=4096 | parallel attn+MLP

In [10]:
import sys
sys.path.insert(0, '../../..')

import matplotlib.pyplot as plt
from src.models import load_model, unload
from src.logit_lens import logit_lens, print_logit_lens
from src.decompose import decompose_contributions, print_decomposition, summarize_contributions
from src.heads import per_head_contributions, top_heads, print_top_heads, print_layer_summary
from src.viz import plot_logit_lens, plot_decomposition, plot_head_heatmap, save_figures

In [11]:
model_name = "pythia-410m"
model, info = load_model(model_name)

Loaded pretrained model pythia-410m into HookedTransformer
Loaded pythia-410m on mps
  24 layers | 16 heads | d_model=1024 | d_mlp=4096 | parallel attn+MLP


In [12]:
prompts = [
    ("The W in WCAG stands for", " Web", "wcag"),
    ("ARIA stands for Accessible Rich Internet", " Applications", "aria"),
    ("The purpose of alt text is to describe an", " image", "alt"),
    ("In HTML, the alt attribute provides a text description of an", " image", "HTML"),
    ("A blind person uses a screen", " reader", "screenReader"),
    ("A screen reader is", " software", "screenReaderv1"),
]

results = {}
for prompt, target, label in prompts:
    print(f"\n{'='*60}")
    print(f"  {label.upper()}: \"{prompt}\" → \"{target}\"")
    print(f"{'='*60}\n")

    df_lens, cache = logit_lens(model, prompt, target)
    print_logit_lens(df_lens)

    df_decomp = decompose_contributions(model, cache, target)
    print_decomposition(df_decomp)
    print(summarize_contributions(df_decomp))

    df_heads = per_head_contributions(model, cache, target)
    print_top_heads(df_heads)

    results[label] = {"lens": df_lens, "decomp": df_decomp, "heads": df_heads}


  WCAG: "The W in WCAG stands for" → " Web"

Model: pythia-410m
Prompt: "The W in WCAG stands for"
Target: " Web" (token 7066)
Final prediction: " Web"

Layer    Top Prediction       Target Rank    Target Prob 
------------------------------------------------------
0        ks                   47334          0.000000    
1        ks                   45265          0.000000    
2        pard                 46814          0.000000    
3                             24376          0.000001    
4        /*!                  16607          0.000002    
5        �                    3775           0.000023    
6        both                 19099          0.000001    
7        switch               21444          0.000001    
8        our                  39111          0.000000    
9        onson                27918          0.000001    
10       all                  25144          0.000001    
11       oded                 40249          0.000000    
12        abbrev              42837  

In [13]:
for label, data in results.items():
    plt.close(plot_logit_lens(data["lens"]))
    plt.close(plot_decomposition(data["decomp"]))
    plt.close(plot_head_heatmap(data["heads"]))

In [14]:
for label, data in results.items():
    data["lens"].to_csv(f"../../results/pythia/{model_name}/{label}-logit-lens.csv", index=False)
    data["decomp"].to_csv(f"../../results/pythia/{model_name}/{label}-decomposition.csv", index=False)
    data["heads"].to_csv(f"../../results/pythia/{model_name}/{label}-heads.csv", index=False)
    save_figures(model_name, label,
                 logit_lens=data["lens"], decomposition=data["decomp"], heads=data["heads"],
                 out_dir=f"../../results/pythia/{model_name}/figures")

Saved: ../../results/pythia/pythia-410m/figures/pythia-410m-wcag-logit-lens.png
Saved: ../../results/pythia/pythia-410m/figures/pythia-410m-wcag-decomposition.png
Saved: ../../results/pythia/pythia-410m/figures/pythia-410m-wcag-head-heatmap.png
Saved: ../../results/pythia/pythia-410m/figures/pythia-410m-aria-logit-lens.png
Saved: ../../results/pythia/pythia-410m/figures/pythia-410m-aria-decomposition.png
Saved: ../../results/pythia/pythia-410m/figures/pythia-410m-aria-head-heatmap.png
Saved: ../../results/pythia/pythia-410m/figures/pythia-410m-alt-logit-lens.png
Saved: ../../results/pythia/pythia-410m/figures/pythia-410m-alt-decomposition.png
Saved: ../../results/pythia/pythia-410m/figures/pythia-410m-alt-head-heatmap.png
Saved: ../../results/pythia/pythia-410m/figures/pythia-410m-HTML-logit-lens.png
Saved: ../../results/pythia/pythia-410m/figures/pythia-410m-HTML-decomposition.png
Saved: ../../results/pythia/pythia-410m/figures/pythia-410m-HTML-head-heatmap.png
Saved: ../../results/py

In [13]:
unload(model)

Model unloaded, memory cleared
